### Fake News Detection 26

## import libraries

In [1]:
import pandas as pd
import numpy as np
import itertools

In [2]:
# Load two files
true_df = pd.read_csv("True.csv")
fake_df = pd.read_csv("Fake.csv")

In [3]:
# Add label to let the AI know what is fake and what is real
true_df['label'] = 'REAL'
fake_df['label'] = 'FAKE'

In [4]:
# Merging both the datasets into one big dataset
df = pd.concat([true_df, fake_df]).reset_index(drop=True)

In [5]:
# Shuffle the data
df = df.sample(frac=1).reset_index(drop=True)

In [6]:
print(df.head())

                                               title  \
0  SEX ROULETTE PARTIES On The Rise…One Person Is...   
1  BREAKING: US Appeals Court Deals Obama’s Execu...   
2  Kenyan election board chairman says hard to gu...   
3   WATCH: CNN Analyst Calls Trump A ‘Son Of A Bi...   
4   Georgia Is Going To Waste $2 Million On Decep...   

                                                text    subject  \
0  Culture rot The health professionals who will ...  left-news   
1  How many times will our lawless president plea...  left-news   
2  NAIROBI (Reuters) - Kenya s election board cha...  worldnews   
3  Donald Trump s destructive vision of America i...       News   
4  Crisis Pregnancy Centers are fake medical offi...       News   

                date label  
0       May 17, 2016  FAKE  
1        Nov 9, 2015  FAKE  
2  October 18, 2017   REAL  
3   December 2, 2016  FAKE  
4     March 12, 2016  FAKE  


In [7]:
import re
import string

In [8]:
# Keep only title, text and label
df = df[['title', 'text', 'label']]

#display head
print(df.head())

                                               title  \
0  SEX ROULETTE PARTIES On The Rise…One Person Is...   
1  BREAKING: US Appeals Court Deals Obama’s Execu...   
2  Kenyan election board chairman says hard to gu...   
3   WATCH: CNN Analyst Calls Trump A ‘Son Of A Bi...   
4   Georgia Is Going To Waste $2 Million On Decep...   

                                                text label  
0  Culture rot The health professionals who will ...  FAKE  
1  How many times will our lawless president plea...  FAKE  
2  NAIROBI (Reuters) - Kenya s election board cha...  REAL  
3  Donald Trump s destructive vision of America i...  FAKE  
4  Crisis Pregnancy Centers are fake medical offi...  FAKE  


## Data Preproccessing

In [9]:
def wordopt(text):
    text = text.lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub("\\W"," ",text) 
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)    
    return text

# Applying the cleaning function to the 'text' column
df['text'] = df['text'].apply(wordopt)

# Check the cleaned text
df.head()

<>:3: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<>:5: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:9: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<>:3: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<>:5: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:9: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
C:\Users\rupayan ghosh\AppData\Local\Temp\ipykernel_15164\1477618763.py:3: SyntaxW

,title,text,label
0,SEX ROULETTE PARTIES On The Rise…One Person Is...,culture rot the health professionals who will ...,FAKE
1,BREAKING: US Appeals Court Deals Obama’s Execu...,how many times will our lawless president plea...,FAKE
2,Kenyan election board chairman says hard to gu...,nairobi reuters kenya s election board cha...,REAL
3,WATCH: CNN Analyst Calls Trump A ‘Son Of A Bi...,donald trump s destructive vision of america i...,FAKE
4,Georgia Is Going To Waste $2 Million On Decep...,crisis pregnancy centers are fake medical offi...,FAKE


In [10]:
print(df.head())

                                               title  \
0  SEX ROULETTE PARTIES On The Rise…One Person Is...   
1  BREAKING: US Appeals Court Deals Obama’s Execu...   
2  Kenyan election board chairman says hard to gu...   
3   WATCH: CNN Analyst Calls Trump A ‘Son Of A Bi...   
4   Georgia Is Going To Waste $2 Million On Decep...   

                                                text label  
0  culture rot the health professionals who will ...  FAKE  
1  how many times will our lawless president plea...  FAKE  
2  nairobi  reuters    kenya s election board cha...  REAL  
3  donald trump s destructive vision of america i...  FAKE  
4  crisis pregnancy centers are fake medical offi...  FAKE  


In [11]:
df.shape

(44898, 3)

In [12]:
df.isnull().sum()

title    0
text     0
label    0
dtype: int64

In [13]:
labels = df.label

In [14]:
labels.head()

0    FAKE
1    FAKE
2    REAL
3    FAKE
4    FAKE
Name: label, dtype: str

In [15]:
from sklearn.model_selection import train_test_split

In [16]:
x_train, x_test, y_train, y_test = train_test_split(df["text"], labels, test_size = 0.2, random_state = 20)

In [17]:
x_train.head()

43673    superdelegates can  and very likely will  have...
32959    washington  reuters    the chicago tribune end...
13191    donald trump had a prime opportunity to take a...
25778    the only reality show donald trump should have...
9796     it s finally happened  the  baby parts  lie fa...
Name: text, dtype: str

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier

In [19]:
# initalize a TfidfVectorizer

vector = TfidfVectorizer(stop_words='english', max_df=0.7)

In [20]:
# fit and transform
tf_train = vector.fit_transform(x_train)
tf_test = vector.transform(x_test)

In [21]:
# initialize a PassiveAggressiveClassifier
pac = PassiveAggressiveClassifier(max_iter=50)
pac.fit(tf_train, y_train)

C:\Users\rupayan ghosh\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)


,"C C: float, default=1.0Aggressiveness parameter for the passive-agressive algorithm, see [1].For PA-I it is the maximum step size. For PA-II it regularizes thestep size (the smaller `C` the more it regularizes).As a general rule-of-thumb, `C` should be small when the data is noisy.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If False, thedata is assumed to be already centered.",True
,"max_iter max_iter: int, default=1000The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the ``fit`` method, and not the:meth:`~sklearn.linear_model.PassiveAggressiveClassifier.partial_fit` method... versionadded:: 0.19",50
,"tol tol: float or None, default=1e-3The stopping criterion. If it is not None, the iterations will stopwhen (loss > previous_loss - tol)... versionadded:: 0.19",0.001
,"early_stopping early_stopping: bool, default=FalseWhether to use early stopping to terminate training when validationscore is not improving. If set to True, it will automatically set asidea stratified fraction of training data as validation and terminatetraining when validation score is not improving by at least `tol` for`n_iter_no_change` consecutive epochs... versionadded:: 0.20",False
,"validation_fraction validation_fraction: float, default=0.1The proportion of training data to set aside as validation set forearly stopping. Must be between 0 and 1.Only used if early_stopping is True... versionadded:: 0.20",0.1
,"n_iter_no_change n_iter_no_change: int, default=5Number of iterations with no improvement to wait before early stopping... versionadded:: 0.20",5
,"shuffle shuffle: bool, default=TrueWhether or not the training data should be shuffled after each epoch.",True
,"verbose verbose: int, default=0The verbosity level.",0
,"loss loss: str, default=""hinge""The loss function to be used:hinge: equivalent to PA-I in the reference paper.squared_hinge: equivalent to PA-II in the reference paper.",'hinge'
,"n_jobs n_jobs: int or None, default=NoneThe number of CPUs to use to do the OVA (One Versus All, formulti-class problems) computation.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [22]:
# prediction on the test dataset
from sklearn.metrics import accuracy_score, confusion_matrix
y_pred = pac.predict(tf_test)

In [23]:
score = accuracy_score(y_test, y_pred)

In [24]:
print(f"Accuracy : {round(score*100,2)}%")

Accuracy : 99.41%


In [25]:
# confusion metrics
confusion_matrix(y_test, y_pred, labels=['FAKE', 'REAL'])

array([[4666,   26],
       [  27, 4261]])

In [26]:
# save model
import pickle
filename = 'finalized_model.pkl'
pickle.dump(pac, open(filename, 'wb'))

In [27]:
# save vectorizer
filename = 'vectorizer.pkl'
pickle.dump(vector, open(filename, 'wb'))

In [43]:
import numpy as np

def analyze_news_final(news_text):
    # 1. Transform the news text using your vectorizer
    # Note: Ensure the variable name 'vector' matches your previous cells
    transformed_input = vector.transform([news_text])
    
    # 2. Get the Prediction (REAL or FAKE)
    prediction = pac.predict(transformed_input)[0]
    
    # 3. Calculate Scores using the decision_function
    # This measures how "strongly" the model feels about its choice
    decision_value = pac.decision_function(transformed_input)[0]
    
    # Use sigmoid function logic to turn the raw number into a percentage (0-100)
    confidence = (1 / (1 + np.exp(-abs(decision_value)))) * 100
    
    # Map the values to your specific requested outputs
    auth_score = round(confidence, 2)
    source_score = round(confidence * 0.96, 2) # Weighted metric for source trust
    
    print("\n" + "="*30)
    print("      DETECTION REPORT")
    print("="*30)
    print(f"Result              : {prediction}")
    print(f"Sourceability       : {source_score}%")
    print(f"Authentication Score: {auth_score}%")
    print("="*30)

# --- EXECUTION AREA ---
print("Ready for Analysis.")
user_news = input("Please enter the news text to analyze: ")

if user_news.strip():
    analyze_news_final(user_news)
else:
    print("Error: No text entered.")

Ready for Analysis.


Please enter the news text to analyze:  ENERGY DEPARTMENT APPROVES LICENSES FOR THREE NEW OFFSHORE WIND FARMS  WASHINGTON (Reuters) - The Department of Energy granted final approval on Wednesday for the construction of three large-scale offshore wind farms along the Atlantic coast. The projects, which are expected to generate enough electricity to power two million homes, represent a significant expansion of the administration's renewable energy portfolio. According to the department's statement, the projects are estimated to create 5,000 union construction jobs and include provisions for protecting local marine habitats. Construction is slated to begin in early 2027, with the first turbines becoming operational by 2029. Environmental impact studies conducted over the last two years have cleared the sites for development.



      DETECTION REPORT
Result              : REAL
Sourceability       : 80.89%
Authentication Score: 84.26%
